# Transfer Learning USAD — Submatriz Canal Único (Sensor 203)

Notebook compatible con Google Colab. Cargado desde GitHub.

**Arquitectura**: USAD (dual-decoder PyTorch) con transfer learning por submatriz.  
**Normalización**: Z-score idéntica al notebook `Monografia_S_RNN_Mask.ipynb`.  
**Datos**: Sensor temperatura 203 SIATA — `data/plan_b/203.csv`.

## Cell 1 — Instalación de dependencias

In [ ]:
!pip install torch torchvision plotly scikit-learn pandas numpy matplotlib requests --quiet

## Cell 2 — Imports

In [ ]:
import io
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import torch
import torch.nn as nn
import torch.utils.data as data_utils
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from sklearn.metrics import (
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix as sk_confusion_matrix,
)

print('Imports OK')

## Cell 3 — Clonar repositorio y configurar módulos

> **Reemplaza `REPO_URL` con la URL de tu repositorio en GitHub.**

In [ ]:
import os, sys

REPO_URL = "https://github.com/ronvas234/data-science-monograph.git"  # <-- reemplazar
REPO_DIR = "data-science-monograph"

# Clonar solo si no existe
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")

# Cambiar al directorio del repo (solo si aún no estamos dentro)
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

# Agregar modelos/usad al sys.path para que `from usad import ...` encuentre utils.py
USAD_MODULE_PATH = os.path.abspath("modelos/usad")
if USAD_MODULE_PATH not in sys.path:
    sys.path.insert(0, USAD_MODULE_PATH)

print(f"CWD: {os.getcwd()}")
print(f"USAD path en sys.path: {USAD_MODULE_PATH}")

# Importar clases de USAD — ahora funciona porque utils.py está en el mismo path
from usad import Encoder, Decoder, UsadModel

print("UsadModel cargado:", UsadModel)

## Cell 4 — LinearDecoder (adaptación para Z-score)

El `Decoder` original de USAD termina con `Sigmoid`, diseñado para datos MinMax [0,1].  
Con normalización Z-score los valores pueden ser negativos, por lo que se reemplaza
la activación final por una capa lineal sin activación.

In [ ]:
class LinearDecoder(nn.Module):
    """Decoder sin Sigmoid final — compatible con datos Z-score (rango irrestricto)."""

    def __init__(self, latent_size: int, out_size: int):
        super().__init__()
        self.linear1 = nn.Linear(latent_size, int(out_size / 4))
        self.linear2 = nn.Linear(int(out_size / 4), int(out_size / 2))
        self.linear3 = nn.Linear(int(out_size / 2), out_size)
        self.relu = nn.ReLU(True)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        out = self.relu(self.linear1(z))
        out = self.relu(self.linear2(out))
        return self.linear3(out)  # salida lineal, sin Sigmoid


class UsadModelLinear(nn.Module):
    """UsadModel con LinearDecoder en lugar del Decoder con Sigmoid."""

    def __init__(self, w_size: int, z_size: int):
        super().__init__()
        self.encoder  = Encoder(w_size, z_size)
        self.decoder1 = LinearDecoder(z_size, w_size)
        self.decoder2 = LinearDecoder(z_size, w_size)

    def training_step(self, batch: torch.Tensor, n: int):
        z  = self.encoder(batch)
        w1 = self.decoder1(z)
        w2 = self.decoder2(z)
        w3 = self.decoder2(self.encoder(w1))
        loss1 = (1 / n) * torch.mean((batch - w1) ** 2) + (1 - 1 / n) * torch.mean((batch - w3) ** 2)
        loss2 = (1 / n) * torch.mean((batch - w2) ** 2) - (1 - 1 / n) * torch.mean((batch - w3) ** 2)
        return loss1, loss2

    def validation_step(self, batch: torch.Tensor, n: int) -> Dict:
        with torch.no_grad():
            z  = self.encoder(batch)
            w1 = self.decoder1(z)
            w2 = self.decoder2(z)
            w3 = self.decoder2(self.encoder(w1))
            loss1 = (1 / n) * torch.mean((batch - w1) ** 2) + (1 - 1 / n) * torch.mean((batch - w3) ** 2)
            loss2 = (1 / n) * torch.mean((batch - w2) ** 2) - (1 - 1 / n) * torch.mean((batch - w3) ** 2)
        return {'val_loss1': loss1, 'val_loss2': loss2}

    def validation_epoch_end(self, outputs: List[Dict]) -> Dict:
        loss1 = torch.stack([x['val_loss1'] for x in outputs]).mean()
        loss2 = torch.stack([x['val_loss2'] for x in outputs]).mean()
        return {'val_loss1': loss1.item(), 'val_loss2': loss2.item()}


print('LinearDecoder y UsadModelLinear definidos')

## Cell 5 — DataConfig

In [ ]:
@dataclass
class DataConfig:
    """Centraliza todos los hiperparámetros — sin comportamiento (S)."""

    # Rutas locales (relativas al directorio del repo clonado en Cell 3)
    csv_path:   str = "modelos/usad/data/plan_b/203.csv"
    model_path: str = "modelos/usad/model.pth"

    # Filtro temporal — idéntico al notebook de referencia (Cell 13)
    date_start: str = "2023-01-01"
    date_end:   str = "2023-06-30"

    # Ventanas — idéntico al notebook de referencia (Cell 21)
    window_size: int   = 30
    stride:      int   = 10
    sentinel:    float = -2000.0

    # Dimensiones del modelo nuevo (canal único)
    w_size: int = 30    # = window_size * 1 canal
    z_size: int = 10    # espacio latente proporcional

    # Dimensiones del modelo preentrenado (SWaT, 612 = 12 pasos × 51 sensores)
    w_size_orig: int = 612
    z_size_orig: int = 120

    # Entrenamiento
    epochs_phase1: int   = 30
    epochs_phase2: int   = 70
    batch_size:    int   = 64
    lr:            float = 1e-3
    alpha:         float = 0.5
    beta:          float = 0.5

    # Dispositivo
    device: str = field(
        default_factory=lambda: "cuda" if torch.cuda.is_available() else "cpu"
    )


print('DataConfig OK')

## Cell 6 — AbstractDataLoader + TimeSeriesLoader

In [ ]:
class AbstractDataLoader(ABC):
    """Abstracción para carga de datos (D — depender de abstracciones)."""

    @abstractmethod
    def load(self) -> pd.DataFrame:
        """Retorna DataFrame indexado por fecha_hora con columnas [t, flag, Split, t_mask]."""
        ...


class TimeSeriesLoader(AbstractDataLoader):
    """Carga CSV desde ruta local, parsea fechas y aplica filtro temporal (S)."""

    def __init__(self, config: DataConfig):
        self._cfg = config

    def load(self) -> pd.DataFrame:
        df = self._read_csv(self._cfg.csv_path)
        df = self._parse_and_index(df)
        df = self._apply_temporal_filter(df)
        return df

    def _read_csv(self, path: str) -> pd.DataFrame:
        return pd.read_csv(path)

    def _parse_and_index(self, df: pd.DataFrame) -> pd.DataFrame:
        # Idéntico al notebook de referencia Cell 13
        df['fecha_hora'] = pd.to_datetime(df['fecha_hora'], format='%Y-%m-%d %H:%M:%S')
        return df.set_index('fecha_hora')

    def _apply_temporal_filter(self, df: pd.DataFrame) -> pd.DataFrame:
        # Idéntico al notebook de referencia Cell 13
        return df[(df.index >= self._cfg.date_start) & (df.index <= self._cfg.date_end)]


print('TimeSeriesLoader OK')

## Cell 7 — ZScoreNormalizer

In [ ]:
class ZScoreNormalizer:
    """Normalización Z-score — solo normaliza (S).
    
    Idéntico al notebook de referencia Cell 19:
      media = data_train[data_train.flag==0]['t'].mean()
      std   = data_train[data_train.flag==0]['t'].std()
    """

    def __init__(self, sentinel: float = -2000.0):
        self.sentinel = sentinel
        self.mean_: Optional[float] = None
        self.std_:  Optional[float] = None

    def fit(self, series: pd.Series, flag: pd.Series) -> 'ZScoreNormalizer':
        """Calcula media/std solo sobre valores normales (flag==0) del conjunto de entrenamiento."""
        valid = series[flag == 0]
        self.mean_ = float(valid.mean())
        self.std_  = float(valid.std())
        return self

    def transform(self, series: pd.Series, t_mask: pd.Series) -> pd.Series:
        """Aplica z-score preservando el sentinel -2000 con np.where.
        
        Para val/test: pasar t_mask=series (sin mascara) — el np.where
        pasa todo directamente porque no hay valores -2000.
        """
        normalized = np.where(
            t_mask == self.sentinel,
            self.sentinel,
            (t_mask - self.mean_) / self.std_,
        )
        return pd.Series(normalized, index=series.index, dtype=np.float32)

    def inverse_transform(self, array: np.ndarray) -> np.ndarray:
        """Desnormaliza: x * std + mean."""
        return array * self.std_ + self.mean_


print('ZScoreNormalizer OK')

## Cell 8 — MaskedWindowDataset + DataLoaderFactory + build_unfiltered_windows

In [ ]:
class AbstractWindowDataset(ABC, data_utils.Dataset):
    """Interfaz mínima para datasets de ventanas (I — segregación de interfaces)."""

    @abstractmethod
    def __len__(self) -> int: ...

    @abstractmethod
    def __getitem__(self, idx: int) -> torch.Tensor: ...


class MaskedWindowDataset(AbstractWindowDataset):
    """Dataset de ventanas deslizantes con filtrado de sentinel (S).
    
    Portar de crear_ventanas() del notebook de referencia (Cell 4),
    adaptado para USAD: shape de salida (window_size,) — plano, no (C,1).
    """

    def __init__(
        self,
        series: np.ndarray,
        window_size: int,
        stride: int,
        sentinel: float = -2000.0,
        filter_masked: bool = True,
    ):
        self._windows = self._build_windows(series, window_size, stride, sentinel, filter_masked)

    def _build_windows(self, series, window_size, stride, sentinel, filter_masked) -> np.ndarray:
        windows = []
        for start in range(0, len(series) - window_size + 1, stride):
            window = series[start : start + window_size]
            if np.isnan(window).any():
                continue
            if filter_masked and np.any(window == sentinel):
                continue
            windows.append(window)
        return np.array(windows, dtype=np.float32)

    def __len__(self) -> int:
        return len(self._windows)

    def __getitem__(self, idx: int) -> torch.Tensor:
        return torch.tensor(self._windows[idx])  # shape: (window_size,)


class DataLoaderFactory:
    """Crea DataLoaders con el formato que espera el loop de USAD (S).
    
    El loop original es: for [batch] in train_loader
    Entonces el DataLoader debe devolver listas de 1 tensor (TensorDataset).
    """

    @staticmethod
    def create(
        dataset: AbstractWindowDataset,
        batch_size: int,
        shuffle: bool = False,
    ) -> data_utils.DataLoader:
        tensor_ds = data_utils.TensorDataset(
            torch.from_numpy(dataset._windows)
        )
        return data_utils.DataLoader(
            tensor_ds, batch_size=batch_size, shuffle=shuffle, num_workers=0
        )


def build_unfiltered_windows(series: np.ndarray, window_size: int, stride: int) -> np.ndarray:
    """Genera todas las ventanas sin filtrar (para val/test e inferencia)."""
    windows = []
    for start in range(0, len(series) - window_size + 1, stride):
        windows.append(series[start : start + window_size])
    return np.array(windows, dtype=np.float32)


print('MaskedWindowDataset, DataLoaderFactory OK')

## Cell 9 — WeightTransferService + FineTuningStrategy

In [ ]:
class WeightTransferService:
    """Extrae submatrices del checkpoint preentrenado e inicializa el modelo nuevo (S + D).
    
    Transfer Learning por submatriz:
      w_size_orig=612 → w_size_new=30
      z_size_orig=120 → z_size_new=10
    
    Se extraen los primeros [:h_new, :w_new] elementos de cada capa, que
    corresponden al bloque de características de los primeros pasos temporales.
    """

    def __init__(self, config: DataConfig):
        self._cfg = config

    def load_checkpoint(self, path: str) -> Dict:
        """Carga model.pth desde ruta local."""
        return torch.load(path, map_location='cpu', weights_only=False)

    def transfer(self, model: UsadModelLinear, checkpoint: Dict) -> UsadModelLinear:
        """Extrae submatrices y carga en el modelo. Retorna el modelo para encadenamiento."""
        cfg = self._cfg
        w   = cfg.w_size   # 30
        z   = cfg.z_size   # 10
        h1  = int(w / 2)   # 15
        h2  = int(w / 4)   # 7

        enc_sd  = self._extract_encoder(checkpoint['encoder'],  w, h1, h2, z)
        dec1_sd = self._extract_decoder(checkpoint['decoder1'], w, h1, h2, z)
        dec2_sd = self._extract_decoder(checkpoint['decoder2'], w, h1, h2, z)

        model.encoder.load_state_dict(enc_sd)
        model.decoder1.load_state_dict(dec1_sd)
        model.decoder2.load_state_dict(dec2_sd)
        return model

    def _extract_encoder(self, sd: Dict, w: int, h1: int, h2: int, z: int) -> Dict:
        """Submatriz del encoder: primeras filas y columnas de cada capa.
        
        orig linear1.weight [306, 612] → new [15, 30]  ([:h1, :w])
        orig linear2.weight [153, 306] → new [7,  15]  ([:h2, :h1])
        orig linear3.weight [120, 153] → new [10,  7]  ([:z,  :h2])
        """
        return {
            'linear1.weight': sd['linear1.weight'][:h1, :w].clone(),
            'linear1.bias':   sd['linear1.bias'][:h1].clone(),
            'linear2.weight': sd['linear2.weight'][:h2, :h1].clone(),
            'linear2.bias':   sd['linear2.bias'][:h2].clone(),
            'linear3.weight': sd['linear3.weight'][:z,  :h2].clone(),
            'linear3.bias':   sd['linear3.bias'][:z].clone(),
        }

    def _extract_decoder(self, sd: Dict, w: int, h1: int, h2: int, z: int) -> Dict:
        """Submatriz del decoder (simétrico al encoder).
        
        orig linear1.weight [153, 120] → new [7,  10]  ([:h2, :z])
        orig linear2.weight [306, 153] → new [15,  7]  ([:h1, :h2])
        orig linear3.weight [612, 306] → new [30, 15]  ([:w,  :h1])
        """
        return {
            'linear1.weight': sd['linear1.weight'][:h2, :z].clone(),
            'linear1.bias':   sd['linear1.bias'][:h2].clone(),
            'linear2.weight': sd['linear2.weight'][:h1, :h2].clone(),
            'linear2.bias':   sd['linear2.bias'][:h1].clone(),
            'linear3.weight': sd['linear3.weight'][:w,  :h1].clone(),
            'linear3.bias':   sd['linear3.bias'][:w].clone(),
        }


class FineTuningStrategy:
    """Gestiona qué parámetros se entrenan en cada fase (S + O — extensible)."""

    @staticmethod
    def freeze_encoder(model: UsadModelLinear) -> None:
        """Fase 1: congela encoder, solo se entrenan los decoders."""
        for param in model.encoder.parameters():
            param.requires_grad = False

    @staticmethod
    def unfreeze_all(model: UsadModelLinear) -> None:
        """Fase 2: descongela todo para fine-tuning completo."""
        for param in model.parameters():
            param.requires_grad = True

    @staticmethod
    def count_trainable(model: UsadModelLinear) -> int:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)


print('WeightTransferService, FineTuningStrategy OK')

## Cell 10 — AbstractTrainer + UsadFineTuner

In [ ]:
class AbstractTrainer(ABC):
    """Interfaz mínima de entrenamiento (I + D)."""

    @abstractmethod
    def fit(
        self,
        train_loader: data_utils.DataLoader,
        val_loader:   data_utils.DataLoader,
        epochs:       int,
    ) -> List[Dict]:
        """Retorna historial [{epoch, val_loss1, val_loss2}]."""
        ...


class UsadFineTuner(AbstractTrainer):
    """Loop de entrenamiento USAD con dos optimizadores Adam (S + D).
    
    Modelo, config y device son inyectados — no se crean internamente.
    Porta la función training() de usad.py con historial estructurado.
    """

    def __init__(
        self,
        model:  UsadModelLinear,
        config: DataConfig,
        device: torch.device,
    ):
        self._model  = model
        self._cfg    = config
        self._device = device

    def fit(
        self,
        train_loader: data_utils.DataLoader,
        val_loader:   data_utils.DataLoader,
        epochs:       int,
    ) -> List[Dict]:
        opt1 = torch.optim.Adam(
            list(self._model.encoder.parameters()) +
            list(self._model.decoder1.parameters()),
            lr=self._cfg.lr,
        )
        opt2 = torch.optim.Adam(
            list(self._model.encoder.parameters()) +
            list(self._model.decoder2.parameters()),
            lr=self._cfg.lr,
        )

        history: List[Dict] = []

        for epoch in range(1, epochs + 1):
            self._model.train()
            for [batch] in train_loader:
                batch = batch.to(self._device)
                # AE1
                loss1, _ = self._model.training_step(batch, epoch)
                loss1.backward()
                opt1.step()
                opt1.zero_grad()
                # AE2
                _, loss2 = self._model.training_step(batch, epoch)
                loss2.backward()
                opt2.step()
                opt2.zero_grad()

            self._model.eval()
            val_result = self._evaluate(val_loader, epoch)
            history.append({'epoch': epoch, **val_result})

            if epoch % 10 == 0:
                print(
                    f"Época [{epoch:3d}/{epochs}]  "
                    f"val_loss1: {val_result['val_loss1']:.4f}  "
                    f"val_loss2: {val_result['val_loss2']:.4f}"
                )

        return history

    def _evaluate(self, val_loader: data_utils.DataLoader, epoch: int) -> Dict:
        outputs = [
            self._model.validation_step(batch.to(self._device), epoch)
            for [batch] in val_loader
        ]
        return self._model.validation_epoch_end(outputs)


print('UsadFineTuner OK')

## Cell 11 — InferenceService

In [ ]:
class InferenceService:
    """Ejecuta inferencia y reconstruye la serie temporal (S + D).
    
    Porta dataset_error_V2() y reconstruir_serie() del notebook de referencia.
    Modelo y normalizer son inyectados.
    """

    def __init__(
        self,
        model:      UsadModelLinear,
        normalizer: ZScoreNormalizer,
        config:     DataConfig,
        device:     torch.device,
    ):
        self._model      = model
        self._normalizer = normalizer
        self._cfg        = config
        self._device     = device

    def compute_errors(self, windows_array: np.ndarray, data_split: pd.DataFrame) -> pd.DataFrame:
        """Infiere, reconstruye serie y calcula error cuadrático por paso.
        
        Porta dataset_error_V2() del notebook de referencia.
        Retorna DataFrame con [t, t_predict, flag, error] indexado por fecha_hora.
        """
        reconstructed_norm = self._reconstruct_series(windows_array)
        reconstructed      = self._normalizer.inverse_transform(reconstructed_norm)

        n = len(reconstructed)
        data_aligned = data_split.iloc[:n].copy()

        df = pd.DataFrame(index=data_aligned.index)
        df['t']         = data_aligned['t'].values
        df['t_predict'] = reconstructed
        df['flag']      = data_aligned['flag'].values
        df['error']     = (df['t'] - df['t_predict']) ** 2
        return df

    def _reconstruct_series(self, windows_array: np.ndarray) -> np.ndarray:
        """Forward pass con decoder1(encoder(batch)), retorna serie en escala normalizada."""
        self._model.eval()
        all_w1: List[np.ndarray] = []

        loader = data_utils.DataLoader(
            data_utils.TensorDataset(torch.from_numpy(windows_array)),
            batch_size=self._cfg.batch_size,
            shuffle=False,
        )
        with torch.no_grad():
            for [batch] in loader:
                batch = batch.to(self._device)
                z  = self._model.encoder(batch)
                w1 = self._model.decoder1(z)
                all_w1.append(w1.cpu().numpy())

        all_w1_np = np.concatenate(all_w1, axis=0)  # (N_ventanas, window_size)
        return self._average_windows(all_w1_np)

    def _average_windows(self, w1: np.ndarray) -> np.ndarray:
        """Reconstrucción por overlap-add con promediado.
        
        Porta reconstruir_serie() del notebook de referencia sin cambios de lógica.
        """
        n_windows, window_size = w1.shape
        stride       = self._cfg.stride
        total_length = stride * (n_windows - 1) + window_size

        serie   = np.zeros(total_length)
        counter = np.zeros(total_length)

        for i in range(n_windows):
            start = i * stride
            serie[start : start + window_size]   += w1[i]
            counter[start : start + window_size] += 1

        return serie / counter


print('InferenceService OK')

## Cell 12 — EvaluationService + VisualizationService

In [ ]:
class EvaluationService:
    """Selección de umbral y métricas de clasificación (S + L — sustituible).
    
    Porta precision_recall_curve_plot() y metics() del notebook de referencia.
    """

    def find_optimal_threshold(self, df: pd.DataFrame) -> float:
        """PR curve + argmax(F1) — idéntico a precision_recall_curve_plot() del referencia."""
        precision, recall, thresholds = precision_recall_curve(df['flag'], df['error'])
        f1  = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-8)
        idx = int(np.argmax(f1))
        return float(thresholds[idx])

    def apply_threshold(self, df: pd.DataFrame, threshold: float) -> pd.DataFrame:
        """Agrega columna flag_pred: 1 donde error > threshold."""
        df = df.copy()
        df['flag_pred'] = np.where(df['error'] > threshold, 1, 0)
        return df

    def compute_metrics(self, df: pd.DataFrame) -> Dict:
        """Accuracy, precision, recall, F1, confusion matrix — idéntico a metics() del referencia."""
        y_true = df['flag']
        y_pred = df['flag_pred']
        return {
            'accuracy':         accuracy_score(y_true, y_pred),
            'precision':        precision_score(y_true, y_pred, zero_division=0),
            'recall':           recall_score(y_true, y_pred, zero_division=0),
            'f1':               f1_score(y_true, y_pred, zero_division=0),
            'confusion_matrix': sk_confusion_matrix(y_true, y_pred),
        }


class VisualizationService:
    """Todas las gráficas — cada método tiene una sola responsabilidad (S + I).
    
    Adaptación directa de las funciones de visualización del notebook de referencia.
    """

    # ── Gráfica 1: serie temporal al cargar datos ─────────────────────────────
    @staticmethod
    def plot_splits(
        data_train: pd.DataFrame,
        data_val:   pd.DataFrame,
        data_test:  pd.DataFrame,
        title: str = 'Sensor 203 — Temperatura (°C)',
    ) -> None:
        """Idéntico a Cell 16 del notebook de referencia."""
        plt.figure(figsize=(11, 3))
        plt.plot(data_train.index, data_train['t'], label='Train', color='tab:blue',  lw=1.2)
        plt.plot(data_val.index,   data_val['t'],   label='Val',   color='orange',     lw=1.2)
        plt.plot(data_test.index,  data_test['t'],  label='Test',  color='tab:red',    lw=1.2)
        all_data = pd.concat([data_train, data_val, data_test])
        anom = all_data[all_data['flag'] == 1]
        plt.plot(anom.index, anom['t'], '.', color='purple', ms=2, label='Anomalía real')
        plt.xlabel('Fecha')
        plt.ylabel('Temperatura (°C)')
        plt.title(title)
        plt.legend(ncols=4, loc='best', fontsize=8)
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ── Gráfica 2: curva de entrenamiento ─────────────────────────────────────
    @staticmethod
    def plot_training_history(history: List[Dict], phase1_epochs: int = 30) -> None:
        """Adapta Cell 34 del referencia para USAD (dos pérdidas + separador de fases)."""
        epochs = [h['epoch'] for h in history]
        loss1  = [h['val_loss1'] for h in history]
        loss2  = [h['val_loss2'] for h in history]

        plt.figure(figsize=(10, 4))
        plt.plot(epochs, loss1, '-x', label='val_loss1 (AE1)', color='tab:blue')
        plt.plot(epochs, loss2, '-x', label='val_loss2 (AE2)', color='orange')
        plt.axvline(x=phase1_epochs, color='gray', linestyle='--', alpha=0.6, label='Fase 1 / Fase 2')
        plt.xlabel('Época')
        plt.ylabel('Loss')
        plt.title('Curva de entrenamiento — USAD Transfer Learning Sensor 203')
        plt.legend()
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

    # ── Gráfica 3: Precision-Recall curve ─────────────────────────────────────
    @staticmethod
    def plot_precision_recall(df: pd.DataFrame) -> float:
        """Idéntico a precision_recall_curve_plot() del notebook de referencia (Cell 9).
        Retorna el umbral óptimo.
        """
        precision, recall, thresholds = precision_recall_curve(df['flag'], df['error'])
        f1  = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-8)
        idx = int(np.argmax(f1))

        recall_opt    = recall[idx]
        precision_opt = precision[idx]
        umbral        = thresholds[idx]

        print(f"Umbral óptimo: {umbral:.4f}")

        plt.figure(figsize=(6, 4))
        plt.plot(recall, precision, label='PR Curve')
        plt.scatter(
            recall_opt, precision_opt, color='red',
            label=f'F1 óptimo (thr={umbral:.4f})',
        )
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision-Recall Curve — USAD TL Sensor 203')
        plt.legend()
        plt.grid()
        plt.show()

        return float(umbral)

    # ── Gráfica 4: error de reconstrucción ────────────────────────────────────
    @staticmethod
    def plot_error_reconstruction(
        df:        pd.DataFrame,
        threshold: float,
        label:     str,
        show_pred: bool = False,
    ) -> None:
        """Idéntico a plot_error_reconstruccion() del notebook de referencia (Cell 5)."""
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=df.index, y=df['error'].tolist(),
            mode='lines', name='Error de reconstrucción',
            line=dict(color='steelblue', width=1),
        ))

        reales = df[df['flag'] == 1]
        fig.add_trace(go.Scatter(
            x=reales.index, y=reales['error'].tolist(),
            mode='markers', name='Anomalías reales',
            marker=dict(color='red', size=6),
        ))

        if show_pred and 'flag_pred' in df.columns:
            pred = df[df['flag_pred'] == 1]
            fig.add_trace(go.Scatter(
                x=pred.index, y=pred['error'].tolist(),
                mode='markers', name='Anomalías detectadas',
                marker=dict(color='orange', size=6, symbol='x'),
            ))

        fig.add_hline(
            y=threshold, line_dash='dash', line_color='black',
            annotation_text=f'Umbral θ = {threshold:.4f}',
            annotation_position='top right',
        )

        fig.update_layout(
            title=f'Errores de reconstrucción — conjunto {label}',
            xaxis_title='Tiempo', yaxis_title='Error',
            template='plotly_white', height=400,
            legend=dict(orientation='h'),
        )
        fig.show()

    # ── Gráfica 5: reconstrucción de la serie ─────────────────────────────────
    @staticmethod
    def plot_series_reconstruction(
        df:        pd.DataFrame,
        show_pred: bool = True,
    ) -> None:
        """Idéntico a plot_series() del notebook de referencia (Cell 10)."""
        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=df.index, y=df['t'].tolist(),
            mode='lines', name='Original',
            line=dict(color='steelblue'),
        ))
        fig.add_trace(go.Scatter(
            x=df.index, y=df['t_predict'].tolist(),
            mode='lines', name='Reconstruida',
            line=dict(color='orange'),
        ))

        mask_real = df['flag'] == 1
        fig.add_trace(go.Scatter(
            x=df.index[mask_real], y=df.loc[mask_real, 't'].tolist(),
            mode='markers', name='Anomalías reales',
            marker=dict(color='red', size=6, symbol='circle'),
        ))

        if show_pred and 'flag_pred' in df.columns:
            mask_pred = df['flag_pred'] == 1
            fig.add_trace(go.Scatter(
                x=df.index[mask_pred], y=df.loc[mask_pred, 't'].tolist(),
                mode='markers', name='Anomalías predichas',
                marker=dict(color='purple', size=6, symbol='x'),
            ))

        fig.update_layout(
            title='Reconstrucción vs Original + Anomalías — USAD TL',
            xaxis_title='Fecha', yaxis_title='Temperatura (°C)',
            template='plotly_white', width=1000, height=400,
        )
        fig.show()

    # ── Gráfica 6: métricas ───────────────────────────────────────────────────
    @staticmethod
    def print_metrics(metrics: Dict) -> None:
        """Idéntico a metics() del notebook de referencia (Cell 11)."""
        print(f"Accuracy:  {metrics['accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall:    {metrics['recall']:.4f}")
        print(f"F1:        {metrics['f1']:.4f}")
        print('\nConfusion Matrix:')
        cm = metrics['confusion_matrix']
        print(pd.DataFrame(
            cm,
            index=['Pred Normal', 'Pred Anomalía'],
            columns=['Real Normal', 'Real Anomalía'],
        ))


print('EvaluationService, VisualizationService OK')

---
# Ejecución
---

## Cell 13 — Cargar datos + Gráfica 1: Serie temporal

In [ ]:
cfg    = DataConfig()
device = torch.device(cfg.device)
print(f'Dispositivo: {device}')

loader  = TimeSeriesLoader(cfg)
df_raw  = loader.load()

print(f'Rango temporal: {df_raw.index.min()} → {df_raw.index.max()}')
print(f'Total muestras: {len(df_raw):,}')
print(f'Distribución Split:\n{df_raw["Split"].value_counts()}')

data_train = df_raw[df_raw['Split'] == 'E'].copy()
data_val   = df_raw[df_raw['Split'] == 'V'].copy()
data_test  = df_raw[df_raw['Split'] == 'T'].copy()

print(f'\nTrain: {len(data_train):,} | Val: {len(data_val):,} | Test: {len(data_test):,}')

# ── Gráfica 1 ─────────────────────────────────────────────────────────────────
VisualizationService.plot_splits(data_train, data_val, data_test)

## Cell 14 — Normalización Z-score

In [ ]:
normalizer = ZScoreNormalizer(sentinel=cfg.sentinel)
normalizer.fit(data_train['t'], data_train['flag'])

print(f'Media train (flag==0): {normalizer.mean_:.2f}°C')
print(f'Std  train (flag==0):  {normalizer.std_:.2f}°C')

# Entrenamiento: usar t_mask (contiene -2000 en anomalías)
train_norm = normalizer.transform(data_train['t'], data_train['t_mask'])

# Val/test: usar t directamente (sin mascara) — idéntico al notebook de referencia
val_norm  = normalizer.transform(data_val['t'],  data_val['t'])
test_norm = normalizer.transform(data_test['t'], data_test['t'])

print(f'\nEjemplo train_norm (primeros 5): {train_norm.values[:5]}')

## Cell 15 — Datasets y DataLoaders

In [ ]:
# Entrenamiento: filtrar ventanas con sentinel (filter_masked=True)
train_dataset = MaskedWindowDataset(
    train_norm.values,
    cfg.window_size,
    cfg.stride,
    sentinel=cfg.sentinel,
    filter_masked=True,
)
print(f'Ventanas entrenamiento (filtradas): {len(train_dataset):,}')

train_loader = DataLoaderFactory.create(train_dataset, cfg.batch_size, shuffle=False)

# Val: sin filtrado
val_dataset = MaskedWindowDataset(
    val_norm.values, cfg.window_size, cfg.stride,
    sentinel=cfg.sentinel, filter_masked=False,
)
val_loader = DataLoaderFactory.create(val_dataset, cfg.batch_size)
print(f'Ventanas validación:               {len(val_dataset):,}')

# Arrays sin filtrar para inferencia (reconstrucción de serie completa)
train_arr_full = build_unfiltered_windows(train_norm.values, cfg.window_size, cfg.stride)
val_arr        = build_unfiltered_windows(val_norm.values,   cfg.window_size, cfg.stride)
test_arr       = build_unfiltered_windows(test_norm.values,  cfg.window_size, cfg.stride)

print(f'Ventanas test (inferencia):        {len(test_arr):,}')

## Cell 16 — Instanciar modelo + Transfer Learning por submatriz

In [ ]:
# Modelo nuevo (canal único)
model = UsadModelLinear(w_size=cfg.w_size, z_size=cfg.z_size).to(device)
print('Arquitectura del modelo nuevo:')
print(model)

# Cargar pesos preentrenados y transferir submatriz
transfer_svc = WeightTransferService(cfg)
checkpoint   = transfer_svc.load_checkpoint(cfg.model_path)
print(f'\nCheckpoint keys: {list(checkpoint.keys())}')

model = transfer_svc.transfer(model, checkpoint)
print(f'\nTransfer learning completado.')
print(f'  Encoder linear1.weight orig:  {checkpoint["encoder"]["linear1.weight"].shape}')
print(f'  Encoder linear1.weight nuevo: {model.encoder.linear1.weight.shape}')

## Cell 17 — Fase 1: Entrenamiento (encoder congelado, 30 épocas)

In [ ]:
strategy = FineTuningStrategy()

strategy.freeze_encoder(model)
print(f'Parámetros entrenables Fase 1: {strategy.count_trainable(model):,}')

trainer_p1 = UsadFineTuner(model, cfg, device)
history_p1 = trainer_p1.fit(train_loader, val_loader, epochs=cfg.epochs_phase1)

print('\nFase 1 completada.')

## Cell 18 — Fase 2: Fine-tuning completo (70 épocas) + Gráfica 2: Curva de entrenamiento

In [ ]:
strategy.unfreeze_all(model)
print(f'Parámetros entrenables Fase 2: {strategy.count_trainable(model):,}')

trainer_p2 = UsadFineTuner(model, cfg, device)
history_p2 = trainer_p2.fit(train_loader, val_loader, epochs=cfg.epochs_phase2)

history = history_p1 + [
    {**h, 'epoch': h['epoch'] + cfg.epochs_phase1} for h in history_p2
]

# ── Gráfica 2 ─────────────────────────────────────────────────────────────────
VisualizationService.plot_training_history(history, phase1_epochs=cfg.epochs_phase1)

## Cell 19 — Inferencia (reconstrucción de series train / val / test)

In [ ]:
infer_svc = InferenceService(model, normalizer, cfg, device)

print('Reconstruyendo series...')
df_train_concat = infer_svc.compute_errors(train_arr_full, data_train)
df_val_concat   = infer_svc.compute_errors(val_arr,        data_val)
df_test_concat  = infer_svc.compute_errors(test_arr,       data_test)

print(f'Train → {len(df_train_concat):,} pasos | error medio: {df_train_concat["error"].mean():.4f}')
print(f'Val   → {len(df_val_concat):,} pasos | error medio: {df_val_concat["error"].mean():.4f}')
print(f'Test  → {len(df_test_concat):,} pasos | error medio: {df_test_concat["error"].mean():.4f}')

## Cell 20 — Gráfica 3: Precision-Recall Curve + umbral óptimo

In [ ]:
# El umbral se calcula sobre el conjunto de validación
umbral = VisualizationService.plot_precision_recall(df_val_concat)
print(f'Umbral seleccionado: {umbral:.6f}')

## Cell 21 — Gráfica 4: Error de reconstrucción

In [ ]:
# Val sin predicciones (exploración)
VisualizationService.plot_error_reconstruction(
    df_val_concat, umbral, label='Validación', show_pred=False
)

# Test con predicciones
VisualizationService.plot_error_reconstruction(
    df_test_concat, umbral, label='Test', show_pred=True
)

## Cell 22 — Aplicar umbral + Gráfica 5: Reconstrucción de la serie

In [ ]:
eval_svc = EvaluationService()

df_test_concat = eval_svc.apply_threshold(df_test_concat, umbral)
df_val_concat  = eval_svc.apply_threshold(df_val_concat,  umbral)

# Train (sin predicciones)
VisualizationService.plot_series_reconstruction(df_train_concat, show_pred=False)

# Val (sin predicciones)
VisualizationService.plot_series_reconstruction(df_val_concat, show_pred=False)

# Test (con predicciones)
VisualizationService.plot_series_reconstruction(df_test_concat, show_pred=True)

## Cell 23 — Gráfica 6: Métricas de clasificación

In [ ]:
print('=== Métricas — Conjunto de Validación ===')
metrics_val = eval_svc.compute_metrics(df_val_concat)
VisualizationService.print_metrics(metrics_val)

print('\n=== Métricas — Conjunto de Test ===')
metrics_test = eval_svc.compute_metrics(df_test_concat)
VisualizationService.print_metrics(metrics_test)